In [5]:
articles = []

for category, base_url in categories.items():

    for page in range(1, 6):  # pages 1 to 5

        if page == 1:
            url = base_url
        else:
            url = base_url + f"page/{page}/"

        response = requests.get(url)
        soup = BeautifulSoup(response.text, "html.parser")

        titles = soup.find_all("h2", class_="title")

        print(category, "page", page, len(titles))

        for title in titles:
            link = title.find("a")

            if link:
                articles.append({
                    "category": category,
                    "title": link.get_text(strip=True),
                    "url": link.get("href")
                })

        time.sleep(1)

df_links = pd.DataFrame(articles)

print(df_links.head())
print(df_links.shape)

Local News page 1 14
Local News page 2 14
Local News page 3 14
Local News page 4 14
Local News page 5 14
Business page 1 14
Business page 2 14
Business page 3 14
Business page 4 14
Business page 5 14
World News page 1 14
World News page 2 14
World News page 3 14
World News page 4 14
World News page 5 14
Sport page 1 14
Sport page 2 14
Sport page 3 14
Sport page 4 14
Sport page 5 14
Africa page 1 14
Africa page 2 14
Africa page 3 14
Africa page 4 14
Africa page 5 14
Opinion page 1 14
Opinion page 2 14
Opinion page 3 14
Opinion page 4 14
Opinion page 5 14
     category                                              title  \
0  Local News  Russia Hails Ethiopia’s Peaceful Election, Cit...   
1  Local News  IGAD Congratulates Ethiopia on Successful Cond...   
2  Local News  Indian PM Narendra Modi Congratulates Prime Mi...   
3  Local News  PM Abiy Says Ethiopia Must “Leap Into the Futu...   
4  Local News  PM Abiy Says Ethiopia’s Development Path Is De...   

                               

In [6]:
df_links = pd.DataFrame(articles)

print(df_links.shape)
print(df_links.head())

(420, 3)
     category                                              title  \
0  Local News  Russia Hails Ethiopia’s Peaceful Election, Cit...   
1  Local News  IGAD Congratulates Ethiopia on Successful Cond...   
2  Local News  Indian PM Narendra Modi Congratulates Prime Mi...   
3  Local News  PM Abiy Says Ethiopia Must “Leap Into the Futu...   
4  Local News  PM Abiy Says Ethiopia’s Development Path Is De...   

                                                 url  
0  https://www.fanamc.com/english/russia-hails-et...  
1  https://www.fanamc.com/english/igad-congratula...  
2  https://www.fanamc.com/english/indian-pm-naren...  
3  https://www.fanamc.com/english/pm-abiy-says-et...  
4  https://www.fanamc.com/english/pm-abiy-says-et...  


In [7]:
df_links = df_links.drop_duplicates(subset="url")

print(df_links.shape)

(324, 3)


In [10]:
import requests
from bs4 import BeautifulSoup

# First article in your dataframe
row = df_links.iloc[0]

response = requests.get(row["url"])
soup = BeautifulSoup(response.text, "html.parser")

# Find the article content
content_div = soup.find(
    "div",
    class_="entry-content clearfix single-post-content"
)

# Check whether it was found
print("CONTENT FOUND:", content_div is not None)

if content_div:
    content = content_div.get_text(" ", strip=True)

    print("\nFIRST 1000 CHARACTERS:\n")
    print(content[:1000])

    print("\nTOTAL CHARACTERS:")
    print(len(content))

CONTENT FOUND: True

FIRST 1000 CHARACTERS:

Addis Ababa, June 23, 2026 (FMC) – Russia has welcomed the successful conduct of Ethiopia’s 7th General Election, describing the process as reflecting broad public support for the government’s efforts to strengthen statehood and maintain domestic stability. In a statement issued on Tuesday, the Russian Foreign Ministry noted that Ethiopia held elections for the federal parliament and regional councils on June 1, 2026. The ministry recalled that official results announced by the National Election Board of Ethiopia on June 21 showed the Prosperity Party securing a decisive victory, winning 438 seats in the House of Peoples’ Representatives, the lower chamber of parliament, as well as an overwhelming majority of seats in regional councils. According to the statement, observer missions from the African Union and the Intergovernmental Authority on Development (IGAD) assessed the election campaign and voting process as peaceful and democratic, rep

In [11]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

test_articles = []

for index, row in df_links.head(10).iterrows():

    print(f"Testing article {index + 1}: {row['title']}")

    response = requests.get(row["url"])
    soup = BeautifulSoup(response.text, "html.parser")

    # Date
    date_tag = soup.find("span", class_="time")
    date = date_tag.get_text(" ", strip=True) if date_tag else ""

    # Content
    content_div = soup.find(
        "div",
        class_="entry-content clearfix single-post-content"
    )

    content = (
        content_div.get_text(" ", strip=True)
        if content_div
        else ""
    )

    test_articles.append({
        "category": row["category"],
        "title": row["title"],
        "date": date,
        "url": row["url"],
        "content": content
    })

df_test = pd.DataFrame(test_articles)

print("\nShape:")
print(df_test.shape)

print("\nColumns:")
print(df_test.columns)

print("\nFirst article:")
print(df_test.iloc[0])

print("\nContent preview:")
print(df_test.iloc[0]["content"][:500])

Testing article 1: Russia Hails Ethiopia’s Peaceful Election, Cites Strong Public Backing for Government Agenda
Testing article 2: IGAD Congratulates Ethiopia on Successful Conduct of 7th General Elections
Testing article 3: Indian PM Narendra Modi Congratulates Prime Minister…
Testing article 4: PM Abiy Says Ethiopia Must “Leap Into the Future”…
Testing article 5: PM Abiy Says Ethiopia’s Development Path Is Defined by Collective and Intergenerational…
Testing article 6: Digital systems are widening access to formal finance for previously excluded citizens, PM Abiy says
Testing article 7: Ethiopia’s Reform Programme Entering Integrated Phase Across Finance, Industry, Agriculture and…
Testing article 8: Green Legacy, GERD and Aviation Expansion Driving Ethiopia’s Reform Trajectory, UNECA Chief Says
Testing article 9: Government, Markets and Communities Must Work Together to Sustain Ethiopia’s Reform Gains — PM Abiy
Testing article 10: AU Commission Chief Commends Ethiopia’s Electoral Pr

In [12]:
df_test = pd.DataFrame(test_articles)

print(df_test.shape)

# Save CSV
df_test.to_csv("fana_test.csv", index=False)

print("CSV saved!")

(10, 5)
CSV saved!


In [17]:
from dash import Dash, dcc, html, Input, Output, dash_table
import pandas as pd

df = pd.read_csv("fana_test.csv")

app = Dash(__name__)

app.layout = html.Div([
    html.H1("Fana News Search Dashboard"),

    html.Label("Search by keyword, title, or content"),
    dcc.Input(
        id="search-box",
        type="text",
        placeholder="Type keyword like election, Abiy, finance...",
        style={"width": "100%", "padding": "10px"}
    ),

    html.Br(),
    html.Br(),

    html.Label("Filter by category"),
    dcc.Dropdown(
        id="category-dropdown",
        options=["All"] + list(df["category"].unique()),
        value="All",
        clearable=False
    ),

    html.Br(),

    dash_table.DataTable(
        id="article-table",
        columns=[
            {"name": "Category", "id": "category"},
            {"name": "Date", "id": "date"},
            {"name": "Title", "id": "title"},
            {"name": "URL", "id": "url"},
        ],
        page_size=10,
        style_table={"overflowX": "auto"},
        style_cell={
            "textAlign": "left",
            "whiteSpace": "normal",
            "height": "auto",
            "padding": "8px"
        }
    )
])

@app.callback(
    Output("article-table", "data"),
    Input("search-box", "value"),
    Input("category-dropdown", "value")
)
def update_table(search_term, category):

    filtered_df = df.copy()

    if category != "All":
        filtered_df = filtered_df[filtered_df["category"] == category]

    if search_term:
        search_term = search_term.lower()

        filtered_df = filtered_df[
            filtered_df["title"].str.lower().str.contains(search_term, na=False) |
            filtered_df["content"].str.lower().str.contains(search_term, na=False) |
            filtered_df["date"].str.lower().str.contains(search_term, na=False)
        ]

    return filtered_df[["category", "date", "title", "url"]].to_dict("records")

app.run(debug=True)

In [19]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

full_articles = []

for index, row in df_links.iterrows():

    # Progress update
    if index % 25 == 0:
        print(f"{index} articles completed out of {len(df_links)}")

    response = requests.get(row["url"])
    soup = BeautifulSoup(response.text, "html.parser")

    # Extract date
    date_tag = soup.find("span", class_="time")
    date = date_tag.get_text(" ", strip=True) if date_tag else ""

    # Extract article content
    content_div = soup.find(
        "div",
        class_="entry-content clearfix single-post-content"
    )

    content = (
        content_div.get_text(" ", strip=True)
        if content_div
        else ""
    )

    # Store article data
    full_articles.append({
        "category": row["category"],
        "title": row["title"],
        "date": date,
        "url": row["url"],
        "content": content
    })

# Create DataFrame
df_articles = pd.DataFrame(full_articles)

print("\nScraping Complete!")
print(df_articles.shape)

# Save CSV
df_articles.to_csv("fana_news_database.csv", index=False)

print("CSV saved as fana_news_database.csv")

# Preview data
print(df_articles.head())

0 articles completed out of 324
25 articles completed out of 324
50 articles completed out of 324
75 articles completed out of 324
125 articles completed out of 324
150 articles completed out of 324
175 articles completed out of 324
200 articles completed out of 324
250 articles completed out of 324
275 articles completed out of 324
300 articles completed out of 324
350 articles completed out of 324
375 articles completed out of 324
400 articles completed out of 324

Scraping Complete!
(324, 5)
CSV saved as fana_news_database.csv
     category                                              title  \
0  Local News  Russia Hails Ethiopia’s Peaceful Election, Cit...   
1  Local News  IGAD Congratulates Ethiopia on Successful Cond...   
2  Local News  Indian PM Narendra Modi Congratulates Prime Mi...   
3  Local News  PM Abiy Says Ethiopia Must “Leap Into the Futu...   
4  Local News  PM Abiy Says Ethiopia’s Development Path Is De...   

              date                                     

In [20]:
print(df_articles.shape)

print(df_articles["category"].value_counts())

print(df_articles["content"].str.len().describe())

(324, 5)
category
Local News    54
Business      54
World News    54
Sport         54
Africa        54
Opinion       54
Name: count, dtype: int64
count      324.000000
mean      3008.919753
std       2991.833511
min        618.000000
25%       1599.000000
50%       2105.500000
75%       3108.500000
max      29573.000000
Name: content, dtype: float64


In [21]:
from dash import Dash, dcc, html, Input, Output, dash_table
import pandas as pd

df = pd.read_csv("fana_news_database.csv")

app = Dash(__name__)

app.layout = html.Div([
    html.H1("Fana News Search Dashboard"),

    html.Label("Search articles"),
    dcc.Input(
        id="search-box",
        type="text",
        placeholder="Search by keyword, title, date, or content...",
        style={"width": "100%", "padding": "10px"}
    ),

    html.Br(),
    html.Br(),

    html.Label("Filter by category"),
    dcc.Dropdown(
        id="category-dropdown",
        options=["All"] + list(df["category"].unique()),
        value="All",
        clearable=False
    ),

    html.Br(),

    html.H3(id="results-count"),

    dash_table.DataTable(
        id="article-table",
        columns=[
            {"name": "Date", "id": "date"},
            {"name": "Category", "id": "category"},
            {"name": "Title", "id": "title"},
            {"name": "URL", "id": "url"},
        ],
        page_size=10,
        row_selectable="single",
        style_table={"overflowX": "auto"},
        style_cell={
            "textAlign": "left",
            "whiteSpace": "normal",
            "height": "auto",
            "padding": "8px"
        }
    ),

    html.Hr(),

    html.Div(id="article-viewer")
])

@app.callback(
    Output("article-table", "data"),
    Output("results-count", "children"),
    Input("search-box", "value"),
    Input("category-dropdown", "value")
)
def update_table(search_term, category):

    filtered_df = df.copy()

    if category != "All":
        filtered_df = filtered_df[filtered_df["category"] == category]

    if search_term:
        search_term = search_term.lower()

        filtered_df = filtered_df[
            filtered_df["title"].str.lower().str.contains(search_term, na=False) |
            filtered_df["content"].str.lower().str.contains(search_term, na=False) |
            filtered_df["date"].str.lower().str.contains(search_term, na=False)
        ]

    count_text = f"Showing {len(filtered_df)} articles"

    return filtered_df[["date", "category", "title", "url", "content"]].to_dict("records"), count_text

@app.callback(
    Output("article-viewer", "children"),
    Input("article-table", "selected_rows"),
    Input("article-table", "data")
)
def show_article(selected_rows, table_data):

    if not selected_rows:
        return "Click an article row to read the full content."

    selected_article = table_data[selected_rows[0]]

    return html.Div([
        html.H2(selected_article["title"]),
        html.P(f"Date: {selected_article['date']}"),
        html.P(f"Category: {selected_article['category']}"),
        html.P(selected_article["content"])
    ])

app.run(debug=True)